# Solutions — Strings & Parsing

Exercise notebook: `01_python_fundamentals/01_strings_and_parsing.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Normalize product SKUs

**Working solution**

In [ ]:
def normalize_sku(raw: str) -> str:
    cleaned = raw.strip().upper().replace("_", " ").replace("-", " ")
    return "-".join(cleaned.split())

**Explanation**

`str.split()` with no argument splits on *runs* of whitespace and drops empties, so converting every separator
to a space and re-joining with `-` collapses duplicates and trims the ends in one move.

**Java-style approach (works, but noisy)**

In [ ]:
def normalize_sku_java_style(raw):
    result = ""
    prev_dash = False
    for ch in raw.strip().upper():
        if ch in (" ", "_", "-"):
            if not prev_dash and result != "":
                result = result + "-"
            prev_dash = True
        else:
            result = result + ch
            prev_dash = False
    if result.endswith("-"):
        result = result[:-1]
    return result

**Complexity:** O(n) time and space.

**Common mistakes**

- Using `split(' ')` (single space) — it keeps empty strings between double spaces.
- Forgetting that `str` is immutable: `raw.strip()` returns a new string, it doesn't change `raw`.

## Exercise 2 — 🟢 Easy / 🟡 Medium · Parse key=value config lines

**Working solution**

In [ ]:
def parse_kv(line: str) -> dict[str, str]:
    result = {}
    for segment in line.split(";"):
        if not segment.strip():
            continue
        key, sep, value = segment.partition("=")
        key = key.strip()
        if not sep or not key:
            raise ValueError(f"invalid segment: {segment.strip()!r}")
        result[key] = value.strip()
    return result

**Explanation**

`partition` always returns a 3-tuple `(head, sep, tail)`; `sep` is empty when the delimiter is missing —
a clean way to detect bad input without exceptions or index checks.

**Complexity:** O(n).

**Common mistakes**

- `segment.split('=')` breaks values that contain `=` (tokens, base64, URLs).
- Catching `ValueError` from tuple unpacking instead of checking `sep` — it hides real bugs.

## Exercise 3 — 🟡 Medium · Mask PII in emails

**Working solution**

In [ ]:
def mask_email(email: str) -> str:
    email = email.strip()
    if email.count("@") != 1:
        raise ValueError(f"invalid email: {email!r}")
    local, _, domain = email.partition("@")
    if not local or not domain:
        raise ValueError(f"invalid email: {email!r}")
    if len(local) <= 2:
        masked = local[0] + "*"
    else:
        masked = f"{local[0]}{'*' * (len(local) - 2)}{local[-1]}"
    return f"{masked}@{domain.lower()}"

**Explanation**

Negative indexing (`local[-1]`) and string repetition (`'*' * n`) replace the index arithmetic you'd write in Java.

**Complexity:** O(n).

**Common mistakes**

- `email.split('@')` silently accepts `a@b@c` if you only take `[0]` and `[1]`.
- Lowercasing the whole address — local parts are technically case-sensitive; only normalize the domain.

## Exercise 4 — 🟠 Hard · Parse structured application log lines

**Working solution**

In [ ]:
import re
import shlex

LOG_PREFIX = re.compile(r"^(?P<ts>\d{4}-\d{2}-\d{2}T[\d:.]+Z) (?P<level>[A-Z]+) \[(?P<service>[\w.-]+)\] (?P<rest>.+)$")
INT_FIELDS = {"status", "latency_ms"}

def parse_log_line(line: str) -> dict | None:
    match = LOG_PREFIX.match(line.strip())
    if not match:
        return None
    record = {"ts": match["ts"], "level": match["level"], "service": match["service"]}
    try:
        tokens = shlex.split(match["rest"])
    except ValueError:          # unbalanced quotes
        return None
    for token in tokens:
        key, sep, value = token.partition("=")
        if not sep:
            return None
        record[key] = int(value) if key in INT_FIELDS and value.isdigit() else value
    return record

**Explanation**

Split the problem in two: a regex for the rigid prefix (named groups read like a schema), and `shlex.split`
for the shell-like tail, which already understands quoting. Pre-compile the regex at module level — it's reused per line.

**Fragile approach (what not to do)**

In [ ]:
def parse_log_line_fragile(line):
    parts = line.split(" ")           # breaks on msg="two words"
    return {"ts": parts[0], "level": parts[1], "service": parts[2][1:-1]}

**Complexity:** O(n) per line. Compiling the regex once matters when parsing 20k+ lines.

**Common mistakes**

- Splitting on spaces — quoted messages contain spaces.
- Calling `int()` on every value — `req=00ab` would crash or silently change meaning.
- Forgetting `$`/`^` anchors so partial garbage matches.

## Challenge — 🔴 Very Hard · Parse database connection strings

**Working solution**

In [ ]:
from urllib.parse import parse_qsl, unquote, urlsplit

DEFAULT_PORTS = {"postgresql": 5432, "mysql": 3306, "redshift": 5439}

def parse_connection_string(dsn: str) -> dict:
    dsn = dsn.strip()
    if dsn.startswith("jdbc:"):
        dsn = dsn[len("jdbc:"):]
    parts = urlsplit(dsn)
    scheme = parts.scheme.lower()
    if not parts.hostname:
        raise ValueError("host is required")
    database = parts.path.lstrip("/")
    if not database:
        raise ValueError("database is required")
    port = parts.port or DEFAULT_PORTS.get(scheme)
    if port is None:
        raise ValueError(f"no default port for scheme {scheme!r}")
    return {
        "scheme": scheme,
        "user": unquote(parts.username) if parts.username else None,
        "password": unquote(parts.password) if parts.password else None,
        "host": parts.hostname,
        "port": port,
        "database": database,
        "params": dict(parse_qsl(parts.query)),
    }

**Explanation**

The standard library already solved URL parsing including edge cases (IPv6 hosts, ports, userinfo).
A Python developer's first instinct should be "is this in the stdlib?" — `urllib.parse`, `shlex`, `csv`, `json`,
`datetime`, `pathlib` cover most parsing jobs.

**Complexity:** O(n).

**Common mistakes**

- Splitting on `:` and `@` manually — passwords can contain both once URL-encoded.
- `parts.port` raises `ValueError` for a non-numeric port; that's acceptable here, but know it happens.
- Forgetting `unquote` so `p%40ss` leaks through.